# 01. DOM fast-path 기초

목표: URL 문자열 포함 여부보다 안전한 결정적 판정기를 만들고 false accept/false reject를 측정합니다. 외부 패키지가 필요하지 않습니다.

In [ ]:
from urllib.parse import parse_qs, urlparse

ALLOWED_HOSTS = {"crm.example.test"}

def evaluate_search_step(step_title: str, dom: dict | None) -> tuple[str, list[str]]:
    """완료, 미완료, 판단 불가와 근거를 함께 반환합니다."""
    if not dom:
        return "unknown", ["DOM summary missing"]
    if "search" not in step_title.lower() and "검색" not in step_title:
        return "unknown", ["unsupported step type"]

    parsed = urlparse(dom.get("url", ""))
    if parsed.hostname not in ALLOWED_HOSTS:
        return "unknown", ["untrusted origin"]

    query = parse_qs(parsed.query).get("q", [""])[0].strip()
    result_count = dom.get("result_count")
    if query and isinstance(result_count, int) and result_count >= 0:
        return "completed", ["trusted query", "result state observed"]
    return "not_completed", ["query or result state missing"]

In [ ]:
cases = [
    ("Search customers", {"url": "https://crm.example.test/find?q=alice", "result_count": 2}, "completed"),
    ("고객 검색", {"url": "https://evil.test/?q=alice", "result_count": 2}, "unknown"),
    ("Search customers", {"url": "https://crm.example.test/find?q="}, "not_completed"),
    ("Create customer", {"url": "https://crm.example.test/new"}, "unknown"),
]

for title, dom, expected in cases:
    decision, evidence = evaluate_search_step(title, dom)
    print(f"{decision:13} expected={expected:13} evidence={evidence}")
    assert decision == expected

## 관찰

`unknown`은 오류가 아니라 신호가 부족하거나 지원하지 않는 단계라는 별도 상태입니다. production에서는 DOM을 client가 위조할 수 있으므로 중요한 credential은 server-side event와 교차 확인해야 합니다.